タイタニック号の生存者を予測するプログラム

決定木回帰モデルを用いる。

google driveをmountする

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


titanic dataを読み込む

titanic dataは、train.csvとtest.csvがある。

train.csvはsurviveのカラムが記載されている。

test.csvはsurviveは記載されていない。test.csvはsurviveを予測するため。

In [21]:
import numpy as np
import pandas as pd

#
# titanic dataを置いたgoogle drive上のpathを記載してください。この直下に.csvファイルがあることを想定しています。
#
dir_path = '/content/drive/MyDrive/IPUT/人工知能/予測/data/titanic/'

#
# 学習データの読み込み(pd.read_csvを使い、表形式でデータの読み込みを行う)
#
df = pd.read_csv(dir_path + 'train.csv')

#
# テストデータの読み込み。テストデータはtrainデータの欠損値を埋める時に用いる
#
test_df = pd.read_csv(dir_path + 'test.csv')

dataの欠損値を補完する必要があります。

titanic dataには欠損値がたくさんあるので、何らかの補完をしておかないと学習ができないから。



In [22]:
#
# 覚えていないので、dfのカラムを確認しておく
#
print(df.columns)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


まず、カラムごとの欠損値がどのくらいあるのか確認しておく。

In [36]:
print("titanic data の欠損値")

print(df.isnull().sum())

# print("Embarked の欠損値=", df.Embarked.isnull().sum())

titanic data の欠損値
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


欠損値を埋める。欠損値があるのはAge, Cabin, Embarkedの３つ。

Cabinは使えるとすごそうだが、ほとんど無いため、せめてAgeとEmbarkedは埋めたい。

年齢は中央値とする。

Cabinデータは学習には使わない。

Embarked（乗り込んだ港）は適当に埋める

In [37]:
#
# 年齢の平均値と中央値を確認するときにはコメント外してちょんまげ
#
# print(df.Age.mean())
# print(df.Age.median())

# 学習データとテストデータを連結する
all_df = pd.concat([df, test_df], ignore_index=True)
# print(fill_df)

#
# 学習データとテストデータが連結されていることを確認。
# 中央値は全体の値から計算したいのでこの手法をとる
#
print(all_df.shape)

#
# 念の為、dfをfill_dfにコピー
#
fill_df = df.copy()

#
# 年齢の中央値を全てのデータから計算
#
age_median=all_df.Age.median()

#
# 年齢の欠損値を、計算しておいた中央値で補完する
#
fill_df.Age = fill_df.Age.fillna(age_median)

#
# Embarked 欠損値の補完：どちらもS（サザンプトン港）としてしまう。
#
fill_df.Embarked = fill_df.Embarked.fillna('S')

#
# 乗船した港の欠損値を再度確認する
#
print("欠損値の確認")
print(fill_df.isnull().sum())
# fill_df.head()

(1309, 12)
欠損値の確認
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


データを学習データとテストデータに分割する。

fill_dfのデータを分割して学習データx_train, y_train, テストデータx_test, y_testを作成する。

EmbarkedとSexをホットエンコードに変換する


In [46]:
#
# train_test_splitをインポート。便利！！
#
from sklearn.model_selection import train_test_split

#
# y_trainにSurvivedを挿入する
#
y_train = fill_df["Survived"]

#
# x_trainから使わないカラムを除く。
# 使わないカラム： Survived（y_trainとしたので不要）, Name, Ticket, Cabin
# ticket, cabinには情報があるかもしれません。titanicの設計図と照らし合わせてください。
# 決定木を使うと、欠損値があっても分類できる場合があります。
#
x_train = fill_df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin'])

#
# 確認したい時はコメント外して
#
# print(x_train)
# print(y_train)


#
# Sex, Embarkedをホットエンコードに変換する
# 機械学習で学習できるようにするため
#
x_train = pd.get_dummies(x_train, columns=['Sex', 'Embarked'], prefix='HE')

#
# x_train, x_testを8:2に分割する。
#
x_train, x_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.2)

#
# 一旦確認
#
print(x_train.head())

     PassengerId  Pclass   Age  SibSp  Parch      Fare  HE_female  HE_male  \
779          780       1  43.0      0      1  211.3375       True    False   
94            95       3  59.0      0      0    7.2500      False     True   
407          408       2   3.0      1      1   18.7500      False     True   
334          335       1  28.0      1      0  133.6500       True    False   
561          562       3  40.0      0      0    7.8958      False     True   

      HE_C   HE_Q  HE_S  
779  False  False  True  
94   False  False  True  
407  False  False  True  
334  False  False  True  
561  False  False  True  


決定木分類を学習する。

In [50]:
# ライブラリのインポート
from sklearn import tree
#
# 決定木モデルを作る
#
model = tree.DecisionTreeClassifier()

#
# 決定木モデルを学習
#
model.fit(x_train, y_train)

#
# 作成した決定木モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測結果を表示
#
print(y_pred)

[0 1 0 0 1 1 0 0 1 0 1 1 0 1 0 0 0 1 0 1 1 0 1 1 0 0 0 0 1 0 0 1 0 0 0 0 0
 1 1 1 0 0 0 1 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 1 0 0 0 0 1 1
 0 1 0 0 0 0 0 0 1 0 1 0 0 1 1 1 1 1 0 0 1 1 1 0 0 0 0 1 0 1 0 0 1 0 1 0 1
 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 1 0 1 0 0 1 1 0 1 0 0 1 1 0 0 0 0 0 1 1 0 1
 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 1 0 1 0 0 0 0]


予測結果y_predと正しい結果y_testを比較して、認識率を計算する

In [52]:
from sklearn.metrics import accuracy_score

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred)

#
# 推定率を表示
#
print("推定率=", accuracy)


認識率： 0.7597765363128491
